# Fluorescence Correlation Spectroscopy (FCS) Analysis - Calibration Data

This notebook processes Time-Tagged Time-Resolved (TTTR) data to generate fluorescence correlation functions. The analysis is designed for PIE (Pulsed Interleaved Excitation) measurements and calculates multiple correlation types:

## Correlation Types Analyzed:
1. **PIE Cross-correlation**: Green prompt ↔ Red delay (species co-diffusion)
2. **FRET Cross-correlation**: Green prompt ↔ Red prompt (FRET efficiency/processes)
3. **Green Auto-correlation**: Green parallel ↔ Green perpendicular (molecular dynamics/diffusion)
4. **Red Prompt Auto-correlation**: Red prompt parallel ↔ perpendicular (FRET-sensitized acceptor emission)
5. **Red Delay Auto-correlation**: Red delay parallel ↔ perpendicular (molecular dynamics/diffusion of directly excited acceptor)

## Output Files:
- `.cor` files: Correlation data compatible with ChiSurf & other analysis software, containing the correlation and the correlation amplitude
- `.png` plots: Visual overview of all correlation curves

Tested with tttrlib 0.25.0

**Author:** Katherina Hemmen ~ Core Unit Fluorescence Imaging ~ RVZ

**Contact:** katherina.hemmen@uni-wuerzburg.de

In [1]:
# Import required libraries for correlation analysis and visualization
import numpy as np  # Numerical operations and array handling
import pylab as p   # Plotting and visualization function
import tttrlib      # Time-tagged time-resolved data analysis library
import glob         # File path pattern matching for batch processing
import os           # Operating system interface for file operations

## DEFINE INPUT DATA PATH

In [9]:
# Specify the directory containing calibration TTTR files
path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_Files/Calibration/*.ptu'

## MEASUREMENT CONFIGURATION

In [3]:
# Channel assignments for PIE-FCS setup
# Note: PIE (Pulsed Interleaved Excitation) uses alternating green and red laser pulses

green_ch_s = [0]  # Green perpendicular polarization detection channel
green_ch_p = [3]  # Green parallel polarization detection channel
red_ch_s = [1]    # Red perpendicular polarization detection channel 
red_ch_p = [2]    # Red parallel polarization detection channel

# PIE time window definitions
# Total laser period = 50 ns, TCSPC resolution = 10 ps → 5000 bins total
# PIE splits each period: first 25 ns (prompt), second 25 ns (delay)
prompt_range = 0, 2500     # First half: green excitation + green/red prompt fluorescence
delay_range = 2500, 5000   # Second half: red excitation + red delayed fluorescence

## CORRELATION ALGORITHM SETTINGS

In [5]:
# Multi-tau correlation algorithm parameters
# These control the time resolution and range of the correlation function
n_casc = 20  # Number of cascades (time decades) - determines maximum correlation time
n_bins = 9   # Bins per cascade - determines time resolution within each decade

# Typical result: ~10^-6 to ~10^2 seconds correlation time range
# Higher n_casc = longer correlation times, higher n_bins = better resolution

## MAIN CORRELATION ANALYSIS LOOP
Process each TTTR file to generate all correlation functions

In [10]:
for file in glob.glob(path):  # Iterate over all matching TTTR files in the Calibration folder
    # FILE LOADING AND HEADER EXTRACTION
    # ==================================
    filename = os.path.abspath(file).split(".ptu")[0]  # Extract filename without extension
    data = tttrlib.TTTR(file)                          # Load TTTR data (reads photon macro/micro times and header)
    
    # Extract timing calibration from file header
    header = data.header  # Convenience handle to the file header metadata
    macro_time_calibration = data.header.macro_time_resolution  # Laser period [seconds]
    micro_time_resolution = data.header.micro_time_resolution   # TCSPC bin size [seconds]
    # Get photon timing arrays
    macro_times = data.macro_times   # Absolute photon arrival times
    micro_times = data.micro_times   # Arrival times within each laser period
    # Calculate PIE timing parameters
    number_of_bins = macro_time_calibration/micro_time_resolution  # Total bins per laser period
    PIE_windows_bins = int(number_of_bins/2)                       # PIE window boundary (50% split)
    
    # PHOTON SELECTION BY CHANNEL
    # ===========================
    # Get photon indices for different correlation combinations
    
    # Combined channels for cross-correlations
    all_green_indices = data.get_selection_by_channel(green_ch_s + green_ch_p)  # All green photons
    all_red_indices = data.get_selection_by_channel(red_ch_s + red_ch_p)        # All red photons
    
    # Individual channels for auto-correlations
    green_indices1 = data.get_selection_by_channel(green_ch_s)  # Green perpendicular
    green_indices2 = data.get_selection_by_channel(green_ch_p)  # Green parallel
    red_indices1 = data.get_selection_by_channel(red_ch_s)      # Red perpendicular
    red_indices2 = data.get_selection_by_channel(red_ch_p)      # Red parallel
    
    # CORRELATION ALGORITHM CONFIGURATION
    # ==================================
    settings = {
        "method": "wahl",   # Use Wahl correlation algorithm (efficient for large datasets)
        "n_bins": n_bins,   # Time resolution parameter
        "n_casc": n_casc,   # Time range parameter
        "make_fine": False  # Don't use micro-time information (standard FCS mode)
    }
    
    # CROSS-CORRELATION ANALYSIS
    # ==========================
    # Extract macro-times for cross-correlation calculations
    t_green = macro_times[all_green_indices]  # Green photon arrival times
    t_red = macro_times[all_red_indices]      # Red photon arrival times
    
    # Extract micro-times for PIE window separation
    mt_green = micro_times[all_green_indices]  # Green micro-times
    mt_red = micro_times[all_red_indices]      # Red micro-times
    
    # CREATE PIE SEPARATION WEIGHTS
    # ============================
    # Binary weights to select photons from specific time windows
    
    # Green prompt weights (first half of laser period only)
    w_gp = np.ones_like(t_green, dtype=float)
    w_gp[np.where(mt_green > PIE_windows_bins)] *= 0.0  # Zero out second half
    
    # Red prompt weights (first half of laser period only)
    w_rp = np.ones_like(t_red, dtype=float)
    w_rp[np.where(mt_red > PIE_windows_bins)] *= 0.0    # Zero out second half
    
    # Red delay weights (second half of laser period only)
    w_rd = np.ones_like(t_red, dtype=float)
    w_rd[np.where(mt_red < PIE_windows_bins)] *= 0.0    # Zero out first half
    
    # CALCULATE CROSS-CORRELATIONS
    # ============================
    
    # 1. PIE Cross-correlation (Green prompt ↔ Red delay)
    PIEcorrelation_curve = tttrlib.Correlator(**settings)  # Initialize multi-tau correlator for PIE cross-correlation
    PIEcorrelation_curve.set_events(t_green, w_gp, t_red, w_rd)
    PIE_time_axis = PIEcorrelation_curve.x_axis          # Correlation time axis
    PIE_amplitude = PIEcorrelation_curve.correlation     # Correlation amplitude
    
    # 2. FRET Cross-correlation (Green prompt ↔ Red prompt)
    FRETcrosscorrelation_curve = tttrlib.Correlator(**settings)  # Initialize correlator for FRET cross-correlation
    FRETcrosscorrelation_curve.set_events(t_green, w_gp, t_red, w_rp)
    FRET_time_axis = FRETcrosscorrelation_curve.x_axis   # Should be identical to PIE_time_axis
    FRET_amplitude = FRETcrosscorrelation_curve.correlation
    
    # AUTO-CORRELATION ANALYSIS
    # =========================
    # Extract macro-times for individual channels (auto-correlation)
    t_green1 = macro_times[green_indices1]   # Green perpendicular times
    t_green2 = macro_times[green_indices2]   # Green parallel times
    t_red1 = macro_times[red_indices1]       # Red perpendicular times
    t_red2 = macro_times[red_indices2]       # Red parallel times
    
    # Extract micro-times for PIE window separation
    mt_green1 = micro_times[green_indices1]  # Green perpendicular micro-times
    mt_green2 = micro_times[green_indices2]  # Green parallel micro-times
    mt_red1 = micro_times[red_indices1]      # Red perpendicular micro-times
    mt_red2 = micro_times[red_indices2]      # Red parallel micro-times
    
    # CREATE CHANNEL-SPECIFIC PIE WEIGHTS
    # ===================================
    
    # Green channel weights (prompt window only)
    w_g1 = np.ones_like(t_green1, dtype=float)
    w_g1[np.where(mt_green1 > PIE_windows_bins)] *= 0.0  # Green perpendicular, prompt only
    w_g2 = np.ones_like(t_green2, dtype=float)
    w_g2[np.where(mt_green2 > PIE_windows_bins)] *= 0.0  # Green parallel, prompt only
    
    # Red prompt channel weights (first half only)
    w_r1p = np.ones_like(t_red1, dtype=float)
    w_r1p[np.where(mt_red1 > PIE_windows_bins)] *= 0.0  # Red perpendicular, prompt
    w_r2p = np.ones_like(t_red2, dtype=float)
    w_r2p[np.where(mt_red2 > PIE_windows_bins)] *= 0.0  # Red parallel, prompt
    
    # Red delay channel weights (second half only)
    w_r1d = np.ones_like(t_red1, dtype=float)
    w_r1d[np.where(mt_red1 < PIE_windows_bins)] *= 0.0  # Red perpendicular, delay
    w_r2d = np.ones_like(t_red2, dtype=float)
    w_r2d[np.where(mt_red2 < PIE_windows_bins)] *= 0.0  # Red parallel, delay
    
    # CALCULATE AUTO-CORRELATIONS
    # ===========================
    # 3. Green Auto-correlation (perpendicular ↔ parallel)
    autocorr_prompt_g = tttrlib.Correlator(**settings)  # Initialize correlator for green auto-correlation
    autocorr_prompt_g.set_events(t_green1, w_g1, t_green2, w_g2)
    
    # 4. Red Prompt Auto-correlation (perpendicular ↔ parallel)
    autocorr_prompt_r = tttrlib.Correlator(**settings)  # Initialize correlator for red prompt auto-correlation
    autocorr_prompt_r.set_events(t_red1, w_r1p, t_red2, w_r2p)
    
    # 5. Red Delay Auto-correlation (perpendicular ↔ parallel)
    autocorr_delay_r = tttrlib.Correlator(**settings)  # Initialize correlator for red delay auto-correlation
    autocorr_delay_r.set_events(t_red1, w_r1d, t_red2, w_r2d)
    
    # Extract correlation results
    ACF_prompt_g_time_axis = autocorr_prompt_g.x_axis       # Green auto-correlation time axis
    ACF_prompt_g_amplitude = autocorr_prompt_g.correlation  # Green auto-correlation amplitude
    
    ACF_prompt_r_time_axis = autocorr_prompt_r.x_axis       # Red prompt auto-correlation time axis
    ACF_prompt_r_amplitude = autocorr_prompt_r.correlation  # Red prompt auto-correlation amplitude
    
    ACF_delay_r_time_axis = autocorr_delay_r.x_axis         # Red delay auto-correlation time axis
    ACF_delay_r_amplitude = autocorr_delay_r.correlation    # Red delay auto-correlation amplitude
    
    # DATA EXPORT AND FORMATTING
    # ==========================
    
    # Convert correlation time axis to milliseconds for standard FCS analysis
    time_axis = PIE_time_axis * macro_time_calibration *1000  # [seconds] → [milliseconds]
    
    # CALCULATE MEASUREMENT STATISTICS
    # ===============================
    # Extract measurement parameters for correlation function normalization
    
    # Get measurement duration from file header
    duration = float(header.tag("TTResult_StopAfter")["value"])  # Duration in milliseconds (from header tag TTResult_StopAfter)
    duration_sec = duration / 1000                               # Convert to seconds
    
    # Count photons in each PIE window for count rate calculations
    all_green_photons = micro_times[all_green_indices]
    nr_of_green_photons = (np.array(np.where(all_green_photons <= PIE_windows_bins), dtype=np.int64)).size
    all_red_photons = micro_times[all_red_indices]
    nr_of_red_p_photons = (np.array(np.where(all_red_photons <= PIE_windows_bins), dtype=np.int64)).size  # Prompt
    nr_of_red_d_photons = (np.array(np.where(all_red_photons > PIE_windows_bins), dtype=np.int64)).size   # Delay
    
    # Calculate average count rates [kHz]
    # Divided by 2 because we're averaging over two detection channels (parallel + perpendicular)
    cr_green_p = nr_of_green_photons / 2 / duration_sec / 1000  # Green prompt count rate
    cr_red_p = nr_of_red_p_photons / 2 / duration_sec / 1000    # Red prompt count rate
    cr_red_d = nr_of_red_d_photons / 2 / duration_sec / 1000    # Red delay count rate
    
    # Calculate average count rates for each correlation type
    avg_cr_PIE = (cr_green_p + cr_red_d)   # PIE: green prompt + red delay
    avg_cr_FRET = (cr_green_p + cr_red_p)  # FRET: green prompt + red prompt
    
    # CREATE COMPATIBILITY COLUMNS
    # ============================
    # Third column filled with zeros for ChiSurf & Kristine software compatibility
    # First two entries contain measurement duration and average count rate
    suren_columnPIE = np.zeros_like(time_axis)
    suren_columnPIE[0] = duration_sec  # Measurement duration
    suren_columnPIE[1] = avg_cr_PIE    # Average count rate for PIE
    
    suren_columnFRET = np.zeros_like(time_axis)
    suren_columnFRET[0] = duration_sec
    suren_columnFRET[1] = avg_cr_FRET
    
    suren_column_gp = np.zeros_like(time_axis)
    suren_column_gp[0] = duration_sec
    suren_column_gp[1] = cr_green_p
    
    suren_column_rp = np.zeros_like(time_axis)
    suren_column_rp[0] = duration_sec
    suren_column_rp[1] = cr_red_p
    
    suren_column_rd = np.zeros_like(time_axis)
    suren_column_rd[0] = duration_sec
    suren_column_rd[1] = cr_red_d
    
    # SAVING THE CORRELATION CURVES
    # ============================ 
    # Each file contains 3 column: (i) the time axis, (ii) the correlation amplitude,
    # (iii) measurement duration & average countrate
    np.savetxt(
        filename + '_PIE.cor',
        np.vstack(
            [time_axis, PIE_amplitude, suren_columnPIE]
        ).T,
        delimiter='\t'
    )
    
    np.savetxt(
        filename + '_FRET.cor',
        np.vstack(
            [time_axis, FRET_amplitude, suren_columnFRET]
        ).T,
        delimiter='\t'
    )
        
    np.savetxt(
        filename + '_gp.cor',
        np.vstack(
            [time_axis, ACF_prompt_g_amplitude, suren_column_gp]
        ).T,
        delimiter='\t'
    )
    
    np.savetxt(
        filename + '_rp.cor',
        np.vstack(
            [time_axis, ACF_prompt_r_amplitude, suren_column_rp]
        ).T,
        delimiter='\t'
    )
        
    np.savetxt(
        filename + '_rd.cor',
        np.vstack(
            [time_axis, ACF_delay_r_amplitude, suren_column_rd]
        ).T,
        delimiter='\t'
    )
    
    # PLOTTING THE CORRELATION CURVES
    # ============================
    p.semilogx(time_axis, PIE_amplitude, label='PIE (gp-rd)')  # Plot PIE cross-correlation on a logarithmic time axis
    p.semilogx(time_axis, ACF_prompt_r_amplitude, label='Red prompt')  # Plot red prompt auto-correlation
    p.semilogx(time_axis, ACF_prompt_g_amplitude, label='Green prompt')  # Plot green prompt auto-correlation
    p.semilogx(time_axis, ACF_delay_r_amplitude, label='Red delay')  # Plot red delay auto-correlation
    p.semilogx(time_axis, FRET_amplitude, label='FRET (gp-rp)')  # Plot FRET cross-correlation
    
    p.ylim(ymin=1)  # Force lower y-limit for better visibility of amplitudes
    p.xlabel('correlation time [ms]')  # Label x-axis in milliseconds
    p.ylabel('correlation amplitude')  # Label y-axis with correlation amplitude
    p.legend()  # Show legend with curve labels
    p.savefig(filename + "_corr.png", dpi=150)  # Save plot to PNG next to the input file
    p.close()  # Close the figure to free memory in batch processing

## ADDITIONAL NOTES: Prerequisites & Data Layout
- Python 3.9+ with tttrlib >= 0.21.9 (tested with 0.21.9).
- Data are expected in a subfolder named `Calibration` relative to this notebook/script.
  - Files: `.ptu` TTTR files from your PIE-FCS measurement.
- The script scans `./Calibration/*.ptu` and processes all matching files.


## ADDITIONAL NOTES: Units and Timing Conventions
- `macro_time_calibration` is the macro-time tick in seconds (time between successive macro-time units recorded by the hardware). Multiplying the correlator x-axis by this value converts correlator bins into seconds.
- `micro_time_resolution` is the TCSPC bin width in seconds.
- `number_of_bins = macro_time_calibration / micro_time_resolution` gives the number of TCSPC bins per laser period.
  - For a 50 ns laser period and 10 ps TCSPC resolution: 50 ns / 10 ps = 5000 bins.
- `PIE_windows_bins = int(number_of_bins/2)` splits the period into two equal windows:
  - Prompt window: bins 0 … `PIE_windows_bins` (typically excitation with the first laser; green here).
  - Delay window: bins `PIE_windows_bins` … end (second laser; red here).
- Weights (w_*) set photons outside the desired window to 0 to ensure proper PIE separation.


## ADDITIONAL NOTES: Count Rates and Division by Two
- Count rates are reported per detection polarization.
- Because each color is detected on two polarization channels (parallel and perpendicular), we divide the total photon counts by 2 to compute an average per polarization channel before converting to kHz:
  - Example: `cr_green_p = nr_of_green_photons / 2 / duration_sec / 1000`.
- Aggregated average count rates used for cross-correlations:
  - PIE cross-correlation average rate: `avg_cr_PIE = cr_green_p + cr_red_d`.
  - FRET cross-correlation average rate: `avg_cr_FRET = cr_green_p + cr_red_p`.


## ADDITIONAL NOTES: .cor File Format
Each `.cor` file is a 3-column, tab-delimited text file:
1. Correlation time axis (ms).
2. Correlation amplitude.
3. Compatibility column for ChiSurf/Kristine (zeros), with the following header-like info encoded in the first two rows:
   - Row 1 (index 0): measurement duration in seconds.
   - Row 2 (index 1): average count rate (kHz) appropriate to the correlation in that file:
     - `_PIE.cor`: `avg_cr_PIE`.
     - `_FRET.cor`: `avg_cr_FRET`.
     - `_gp.cor`: `cr_green_p`.
     - `_rp.cor`: `cr_red_p`.
     - `_rd.cor`: `cr_red_d`.
This layout preserves compatibility with external analysis tools that expect a 3rd column.


## Troubleshooting and Tips
- Few or flat correlation curves:
  - Verify channel mapping (green_ch_s/p, red_ch_s/p) matches your acquisition.
  - Check that the PIE windows reflect your instrument timing; adjust prompt/delay split if needed.
- Unexpected count rates:
  - Confirm the acquisition duration read from the header (`TTResult_StopAfter`) is correct for your files.
  - Ensure your data truly contain both polarization channels; if not, revise the division by 2 logic.
- Performance and range:
  - Increase `n_casc` for longer correlation times; increase `n_bins` for finer resolution.
  - Larger values increase computation time and memory usage.
- Visualization:
  - The saved `_corr.png` plot uses a semilog x-axis with a lower y-limit of 1. Adjust labels/limits as needed.
